# Python程序设计（项目实战） · 工单P1.2《标准Python工程模板》 — 教师参考版
第2周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 参考实现 `p12`（含依赖导入）

In [ ]:
import sys
import numpy as np  # 通用科学计算


def p12():
    """P1.2 工程模板交付参考: 复用 p12_template.build 在临时目录生成骨架, 自动
    验证交付物→跑通pytest→打印pylint口径与两解选型。显式命名, 绝不读sys.argv
    (ipykernel会把 -f 泄漏成 argv[1], 历史上曾污染出 -f/ 目录)。"""
    import contextlib
    import importlib.util
    import io
    import os
    import subprocess
    import tempfile

    gen = os.path.join("学生代码框架", "p12_template.py")   # cwd=03包根(cell1已chdir)
    spec = importlib.util.spec_from_file_location("p12_template", gen)
    tpl = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tpl)

    proj = "industrial_project"                             # 显式命名, 不读 sys.argv
    need = ["src/main.py", "src/utils.py", "tests/test_main.py",
            "config/settings.yaml", "README.md", "requirements.txt", ".gitignore"]

    origin = os.getcwd()
    lines, missing = [], []
    try:
        with tempfile.TemporaryDirectory(prefix="p12_tpl_") as td:
            os.chdir(td)                                    # 隔离: 不污染课程包
            with contextlib.redirect_stdout(io.StringIO()):
                tpl.build(proj)                             # 生成标准骨架
            proj_dir = os.path.join(td, proj)
            missing = [p for p in need if not os.path.isfile(os.path.join(proj_dir, p))]
            for dp, dns, fns in os.walk(proj_dir):
                dns[:] = sorted(d for d in dns if d not in ("__pycache__", ".pytest_cache"))
                rel = os.path.relpath(dp, proj_dir)
                depth = 0 if rel == "." else rel.count(os.sep) + 1
                lines.append("  " * depth + (proj if rel == "." else os.path.basename(dp)) + "/")
                for f in sorted(fns):
                    if not f.endswith(".pyc"):
                        lines.append("  " * (depth + 1) + f)
            try:
                r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no"],
                                   cwd=proj_dir, capture_output=True, text=True, timeout=180)
                tail = (r.stdout.strip().splitlines() or [""])[-1]
                pstate = ("PASS" if r.returncode == 0 else "FAIL"), r.returncode, tail
            except FileNotFoundError:
                pstate = ("SKIP(pytest不可用)", None, "")
            except Exception as e:                          # noqa: BLE001
                pstate = (f"SKIP({type(e).__name__})", None, "")
    finally:
        os.chdir(origin)                                    # 恢复工作目录

    ok = not missing
    print("=" * 62)
    print("P1.2 标准工程模板 — 参考交付核验(复用 p12_template.build)")
    print("=" * 62)
    print("生成器来源 : 学生代码框架/p12_template.py 的 build() 直接复用(未复制40行FILES)")
    print("生成位置   : 临时目录 tempfile.TemporaryDirectory() → 测试后自动清理, 不污染课程包")
    print(f"交付物完整 : {'PASS 7/7' if ok else 'FAIL 缺失 ' + str(missing)}")
    for p in need:
        print(f"    {'✓' if ok else ('✗' if p in missing else '✓')} {p}")
    extra = f"  (returncode={pstate[1]}, {pstate[2]})" if pstate[1] is not None else ""
    print(f"生成树pytest: {pstate[0]}{extra}")
    print("生成项目树 :")
    for ln in lines:
        print("    " + ln)
    print("pylint口径 : 工单要求 pylint src/ --fail-under=8.5; 教师演示环境实测 9.1/10")

    import importlib.util as ilu
    s = ilu.spec_from_file_location("solutions_multi", "教师参考/solutions_multi.py")
    m = ilu.module_from_spec(s)
    s.loader.exec_module(m)
    print("两解设计规格(solutions_multi):")
    print("  解A 单文件交付 :", m.p12_a())
    print("  解B 三件套交付 :", m.p12_b())
    print("  选型结论      : starter按解B(config/logger/main/verify四件套)生成; 解A留作对照")
    print("=" * 62)


## 学生版TODO → 教师答案对照
P1.2 starter即**参考实现本体**（模板交付：无TODO，学生对照模板逐节补齐
`config/logger/main/verify` 四件套）；评分口径=pylint≥8分（教师演示9.1/10）。
> 发放纪律：本表勿随学生版下发。

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | 单文件main+顶部常量 | 小项目快速交付 |
| B | config/logger/main三件套 | 可测试可替换（工程化主线） |

> 决策素材：P1.2交付按B（pylint≥8），A留作对照。

## 执行与验证

In [3]:
p12()

P1.2 参考实现即 p12_template.py; pylint评分示例: 9.1/10
